# DeepfakeFusionModel V5 — Temporal rPPG Tokens + True Multi-Head Cross-Attention

**Paper**: *A Hybrid Spatial-Physiological Approach for Deepfake Detection using Swin Transformer and rPPG*

### V5 fixes the softmax-collapse bug found in V4

V4's `CrossAttentionFusion` used `Q ∈ R^(B×1×768)` and `K=V ∈ R^(B×1×768)` — sequence
length 1 on **both** sides. With $L_q=L_k=1$, `Softmax([s]) = [1]` for **any** value of
$s$ — a mathematical identity, not an approximation. The attention output collapsed to a
fixed linear projection of the bio features, completely ignoring the spatial query, and
`W^Q`/`W^K` received **zero gradient** from the task loss.

**Root cause**: the rPPG branch pooled all temporal information into a single 64-d vector
*before* the fusion step, leaving no sequence for attention to attend over.

### What changed in V5

| Component | V4 (broken) | V5 (fixed) |
|---|---|---|
| `rPPGNetwork3D` output | `(B, 64)` — single pooled vector | `(B, BIO_TOKENS=4, 64)` — **sequence of 4 temporal tokens** |
| Cross-Attention K/V | `(B, 1, 768)` — 1 token | `(B, 4, 768)` — **4 tokens**, softmax is a real distribution |
| Attention weights | mathematically forced to `[1.0]` | genuine values that sum to 1 across 4 temporal positions, **inspectable** |
| Gradient to `W^Q`, `W^K` | zero (dead weights) | non-zero — these layers can actually learn |
| Empirical evidence | none | Section 6.5 visualizes attention weights per class (Real vs Fake) |

### Architecture Overview

```
RGB frame (middle of clip)  ──► Swin-Tiny (pretrained) ──► 768-d spatial vector
                                                                 │ (1 query token)
                                                    CrossAttentionFusion (4 heads)
                                                    Q  = spatial            (B, 1, 768)
                                                    K=V= projected bio tokens (B, 4, 768)
                                                    softmax over 4 temporal positions ← REAL
                                                                 │
YCrCb clip (T=8 frames)  ──► 3D CNN rPPGNet ──► 4×64-d temporal tokens ──┘
                              Block1: spatial pool only (T preserved = 8)
                              Block2: spatial + temporal pool (T: 8→4)
                              Block3: spatial-only AdaptiveAvgPool3d((4,1,1))
```

### Notebook Sections
0. Environment Setup
1. Dataset Preparation — Video → Face **Clips** (temporal sequences, `.npy`)
2. Dataset Class & DataLoader
3. Model Architecture V5 (temporal rPPG tokens + true cross-attention)
4. Training Configuration
5. Training Loop
6. Evaluation — FF++ Validation
6.5. **Cross-Attention Weight Analysis** (new — empirical proof attention is non-trivial)
7. Cross-Dataset Evaluation — Celeb-DF-v2
8. Ablation Study (fair comparison)
9. Export & Save


---
## Section 0: Environment Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q timm mediapipe scikit-learn matplotlib seaborn tqdm scipy

In [ ]:
import os, glob, random, time, warnings, gc
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image
import timm
import mediapipe as mp
from sklearn.metrics import (
    accuracy_score, roc_auc_score, roc_curve,
    confusion_matrix, classification_report,
    f1_score, precision_score, recall_score
)
from scipy.optimize import brentq
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from collections import defaultdict

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU:   {torch.cuda.get_device_name(0)}')
    print(f'VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
class Config:
    # ── Paths ──
    DRIVE_ROOT   = '/content/drive/MyDrive/DoAn_Nhom4'
    FF_ZIP       = f'{DRIVE_ROOT}/FaceForensics.zip'
    CELEB_ZIP    = f'{DRIVE_ROOT}/Celeb-DF-v2.zip'
    EXTRACT_DIR  = '/content/dataset'
    CLIPS_DIR    = '/content/clips'
    FF_CLIPS     = f'{CLIPS_DIR}/ffpp'
    CELEB_CLIPS  = f'{CLIPS_DIR}/celeb_df_v2'
    SAVE_DIR     = f'{DRIVE_ROOT}/weights_v5'

    # ── Clip Extraction ──
    # T=8 frames @ 25fps ≈ 320ms window — đủ để capture 1/3 chu kỳ tim (60bpm)
    T                 = 8
    CLIPS_PER_VIDEO   = 4     # clips trích từ mỗi video
    FACE_CONFIDENCE   = 0.7
    FACE_PAD_X        = 0.15
    FACE_PAD_Y_TOP    = 0.40
    FACE_PAD_Y_BOTTOM = 0.20
    FACE_SIZE         = 112   # face crop resolution (px)

    # ── Model ──
    SPATIAL_DIM = 768    # Swin-Tiny output dim
    BIO_DIM     = 64     # rPPG 3D CNN per-token feature dim
    BIO_TOKENS  = T // 2 # NEW: số temporal token rPPG giữ lại cho Cross-Attention (=4)
                          # Đây là điểm sửa cốt lõi: T_out > 1 để softmax không suy biến.
    ATTN_HEADS  = 4       # truyền vào nn.MultiheadAttention
    DROPOUT     = 0.3

    # ── Training ──
    # BATCH_SIZE giảm xuống 8 vì mỗi sample giờ chứa T frames
    BATCH_SIZE           = 8
    EPOCHS               = 25
    LR_SWIN              = 2e-5   # pretrained backbone: fine-tune nhẹ
    LR_OTHER             = 1e-4   # các layer mới: học nhanh hơn
    WEIGHT_DECAY         = 1e-4
    LABEL_SMOOTHING      = 0.05
    EARLY_STOP_PATIENCE  = 5
    GRAD_CLIP            = 1.0
    ABLATION_EPOCHS      = 10     # epochs cho ablation study

    SEED = 42


def set_seed(seed=Config.SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed()
for k, v in vars(Config).items():
    if not k.startswith('_'):
        print(f'  {k:<22} = {v}')


---
## Section 1: Dataset Preparation — Video → Face Clips

**Thay đổi quan trọng so với V3:**  
Thay vì trích xuất frame riêng lẻ, mỗi clip = **T frame liên tiếp** từ cùng một video,
lưu dưới dạng `.npy` shape `(T, FACE_SIZE, FACE_SIZE, 3)` (BGR).  
Điều này cho phép nhánh rPPG phân tích biến động màu sắc theo thời gian.

> **Note:** Chỉ cần chạy Section 1 một lần. Nếu clips đã tồn tại, bỏ qua đến Section 2.

In [ ]:
os.makedirs(Config.EXTRACT_DIR, exist_ok=True)

# Unzip FaceForensics++
ff_folders = glob.glob(os.path.join(Config.EXTRACT_DIR, 'FaceForensics*'))
if not ff_folders:
    print('Extracting FaceForensics++...')
    !unzip -q "{Config.FF_ZIP}" -d "{Config.EXTRACT_DIR}/"
    ff_folders = glob.glob(os.path.join(Config.EXTRACT_DIR, 'FaceForensics*'))
FF_ROOT = ff_folders[0]
print(f'FF++ root: {FF_ROOT}')
!ls "{FF_ROOT}"

# Unzip Celeb-DF-v2
celeb_folders = glob.glob(os.path.join(Config.EXTRACT_DIR, 'Celeb*'))
if not celeb_folders:
    print('\nExtracting Celeb-DF-v2...')
    !unzip -q "{Config.CELEB_ZIP}" -d "{Config.EXTRACT_DIR}/"
    celeb_folders = glob.glob(os.path.join(Config.EXTRACT_DIR, 'Celeb*'))
CELEB_ROOT = celeb_folders[0]
# Handle possible nested folder structure
if not any(os.path.isdir(os.path.join(CELEB_ROOT, d))
           for d in ['Celeb-real', 'Celeb-synthesis']):
    for s in glob.glob(os.path.join(CELEB_ROOT, '*')):
        if os.path.isdir(s) and 'Celeb-real' in os.listdir(s):
            CELEB_ROOT = s
            break
print(f'Celeb-DF-v2 root: {CELEB_ROOT}')
!ls "{CELEB_ROOT}" 

In [ ]:
def extract_clips_from_video(video_path, output_dir, face_detector):
    '''
    Trích xuất face clips từ video.

    Chiến lược:
      1. Đọc frame đầu để detect vị trí mặt (MediaPipe short-range model).
      2. Dùng bounding box cố định cho toàn video (giả định mặt không di chuyển
         nhiều trong video deepfake — hợp lý với FF++ và Celeb-DF).
      3. Tại CLIPS_PER_VIDEO vị trí trải đều trong video, đọc T frame liên tiếp.
      4. Lưu mỗi clip: np.save → shape (T, FACE_SIZE, FACE_SIZE, 3) BGR uint8.

    Dùng cap.grab() để skip frame (KHÔNG decode pixel) → nhanh hơn cap.read() ~10x.
    '''
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return 0

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total < Config.T:
        cap.release()
        return 0

    video_id = os.path.splitext(os.path.basename(video_path))[0]
    os.makedirs(output_dir, exist_ok=True)

    # ── Step 1: Detect face in frame 0 ──────────────────────────────────────
    ret, first_frame = cap.read()
    if not ret:
        cap.release()
        return 0

    rgb = cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB)
    results = face_detector.process(rgb)
    if not results.detections:
        cap.release()
        return 0

    det = results.detections[0]
    bb  = det.location_data.relative_bounding_box
    ih, iw = first_frame.shape[:2]
    x, y = int(bb.xmin * iw), int(bb.ymin * ih)
    w, h = int(bb.width * iw), int(bb.height * ih)

    px = int(w * Config.FACE_PAD_X)
    pt = int(h * Config.FACE_PAD_Y_TOP)
    pb = int(h * Config.FACE_PAD_Y_BOTTOM)
    x1 = max(0, x - px);  y1 = max(0, y - pt)
    x2 = min(iw, x + w + px); y2 = min(ih, y + h + pb)

    if (x2 - x1) < 20 or (y2 - y1) < 20:
        cap.release()
        return 0

    # ── Step 2: Trích clips tại CLIPS_PER_VIDEO vị trí ─────────────────────
    max_start = total - Config.T
    if max_start <= 0:
        cap.release()
        return 0

    clip_starts = np.linspace(0, max_start, Config.CLIPS_PER_VIDEO, dtype=int)
    saved = 0

    for clip_idx, start in enumerate(clip_starts):
        # Seek đến frame bắt đầu
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(start))

        frames = []
        for _ in range(Config.T):
            ret, frame = cap.read()
            if not ret:
                break
            crop = frame[y1:y2, x1:x2]
            if crop.size == 0:
                break
            crop = cv2.resize(crop, (Config.FACE_SIZE, Config.FACE_SIZE),
                              interpolation=cv2.INTER_LINEAR)
            frames.append(crop)

        if len(frames) == Config.T:  # Chỉ lưu clip hoàn chỉnh
            clip_array = np.stack(frames, axis=0)   # (T, H, W, 3) uint8
            out_path = os.path.join(output_dir,
                                    f'{video_id}_clip{clip_idx:02d}.npy')
            np.save(out_path, clip_array)
            saved += 1

    cap.release()
    return saved


def process_video_folders(video_dirs, output_dir):
    '''Xử lý tất cả video trong danh sách thư mục, lưu clips vào output_dir.'''
    # model_selection=0: short-range model, tối ưu cho close-up video (FF++, Celeb-DF)
    # Chính xác hơn và nhanh hơn model_selection=1 (full-range) với close-up faces
    mp_face = mp.solutions.face_detection
    detector = mp_face.FaceDetection(
        model_selection=0,
        min_detection_confidence=Config.FACE_CONFIDENCE
    )

    videos = []
    for vdir in video_dirs:
        if not os.path.isdir(vdir):
            continue
        for ext in ['*.mp4', '*.avi', '*.mov', '*.mkv']:
            videos.extend(glob.glob(os.path.join(vdir, '**', ext), recursive=True))

    print(f'  Found {len(videos)} videos')
    total_clips = 0
    for vp in tqdm(videos, desc='  Extracting clips'):
        total_clips += extract_clips_from_video(vp, output_dir, detector)

    detector.close()
    n_clips = len(glob.glob(os.path.join(output_dir, '*.npy')))
    print(f'  Saved {n_clips} clips total.')
    return n_clips

print('Clip extraction functions ready.')

In [ ]:
# ── Extract FF++ clips ─────────────────────────────────────────────────────────
print('=' * 60)
print('EXTRACTING CLIPS FROM FACEFORENSICS++ (C23)')
print('=' * 60)

ff_real_dir = os.path.join(Config.FF_CLIPS, 'real')
ff_fake_dir = os.path.join(Config.FF_CLIPS, 'fake')

if os.path.exists(ff_real_dir) and len(glob.glob(f'{ff_real_dir}/*.npy')) > 100:
    n_real = len(glob.glob(f'{ff_real_dir}/*.npy'))
    n_fake = len(glob.glob(f'{ff_fake_dir}/*.npy'))
    print(f'FF++ clips already exist — Real: {n_real}, Fake: {n_fake}. Skipping.')
else:
    print('\n[REAL] original/')
    process_video_folders([os.path.join(FF_ROOT, 'original')], ff_real_dir)

    print('\n[FAKE] 5 manipulation methods:')
    for method in ['Deepfakes', 'Face2Face', 'FaceShifter', 'FaceSwap', 'NeuralTextures']:
        mdir = os.path.join(FF_ROOT, method)
        if os.path.isdir(mdir):
            print(f'  >> {method}')
            process_video_folders([mdir], ff_fake_dir)
        else:
            print(f'  >> {method} NOT FOUND — skipping.')

print(f'\nFF++ done: Real={len(glob.glob(f"{ff_real_dir}/*.npy"))} clips, '
      f'Fake={len(glob.glob(f"{ff_fake_dir}/*.npy"))} clips')

In [ ]:
# ── Extract Celeb-DF-v2 clips ─────────────────────────────────────────────────
print('=' * 60)
print('EXTRACTING CLIPS FROM CELEB-DF-v2')
print('=' * 60)

celeb_real_dir = os.path.join(Config.CELEB_CLIPS, 'real')
celeb_fake_dir = os.path.join(Config.CELEB_CLIPS, 'fake')

if os.path.exists(celeb_real_dir) and len(glob.glob(f'{celeb_real_dir}/*.npy')) > 100:
    print('Celeb-DF-v2 clips already exist. Skipping.')
else:
    print('[REAL] Celeb-real + YouTube-real')
    process_video_folders(
        [os.path.join(CELEB_ROOT, 'Celeb-real'),
         os.path.join(CELEB_ROOT, 'YouTube-real')], celeb_real_dir)

    print('[FAKE] Celeb-synthesis')
    process_video_folders([os.path.join(CELEB_ROOT, 'Celeb-synthesis')], celeb_fake_dir)

print(f'Celeb-DF-v2 done: Real={len(glob.glob(f"{celeb_real_dir}/*.npy"))} clips, '
      f'Fake={len(glob.glob(f"{celeb_fake_dir}/*.npy"))} clips')

---
## Section 2: Dataset Class & DataLoader

**Thiết kế dataset cho temporal input:**
- Mỗi sample = 1 clip `.npy` có shape `(T, H, W, 3)` BGR
- **Spatial branch**: middle frame (index T//2) → RGB → Swin transform
- **Bio branch**: toàn bộ T frames → YCrCb → normalize [0,1] → shape `(T, 3, 64, 64)`
- Geometric augmentation áp **nhất quán** cho tất cả T frames (cùng flip/rotate)
- ColorJitter chỉ áp cho spatial branch:
  lý do — nhánh bio cần giữ nguyên màu sắc YCrCb để model học temporal color variation

In [ ]:
class DeepfakeDataset(Dataset):
    '''
    Temporal dual-stream dataset for deepfake detection.

    Load pre-extracted face clips (.npy, shape: T × H × W × 3 BGR).
    Returns:
        spatial_input : (3, 224, 224) float   — ImageNet-normalized RGB middle frame
        bio_input     : (T, 3, 64, 64) float  — YCrCb clip, normalized [0, 1]
        label         : scalar float           — 0=real, 1=fake
    '''
    BIO_SIZE = 64  # spatial size for rPPG input (low-res, cần thông tin màu sắc)

    def __init__(self, npy_paths, labels, spatial_tf, augment=False):
        self.paths     = npy_paths
        self.labels    = labels
        self.spatial_tf = spatial_tf
        self.augment   = augment

    def __len__(self):
        return len(self.paths)

    def _augment_consistent(self, frames_bgr):
        '''
        Áp augmentation geometric lên TẤT CẢ frames với cùng random state.
        QUAN TRỌNG: phải dùng cùng transform cho mỗi frame để đảm bảo
        temporal coherence — rPPG signal phụ thuộc vào consistency theo thời gian.
        '''
        do_flip   = random.random() > 0.5
        do_rotate = random.random() > 0.5
        angle = random.uniform(-10, 10) if do_rotate else 0.0

        h, w = frames_bgr[0].shape[:2]
        M = (cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
             if do_rotate else None)

        out = []
        for f in frames_bgr:
            if do_flip:
                f = cv2.flip(f, 1)
            if do_rotate:
                f = cv2.warpAffine(f, M, (w, h),
                                   borderMode=cv2.BORDER_REPLICATE)
            out.append(f)
        return out

    def __getitem__(self, idx):
        clip_bgr = np.load(self.paths[idx], allow_pickle=False)  # (T, H, W, 3)
        label    = self.labels[idx]

        frames = [clip_bgr[t] for t in range(Config.T)]

        # Geometric augmentation — nhất quán cho tất cả frames
        if self.augment:
            frames = self._augment_consistent(frames)

        mid = Config.T // 2  # index frame giữa clip

        # ── Spatial Branch: RGB middle frame → Swin ─────────────────────────
        frame_rgb = cv2.cvtColor(frames[mid], cv2.COLOR_BGR2RGB)
        img_pil   = Image.fromarray(frame_rgb)
        if self.augment:
            # ColorJitter chỉ cho spatial — KHÔNG áp cho bio vì sẽ làm nhiễu
            # tín hiệu màu sắc temporal mà rPPG net cần học
            img_pil = transforms.ColorJitter(
                brightness=0.15, contrast=0.15, saturation=0.1)(img_pil)
        spatial_input = self.spatial_tf(img_pil)  # (3, 224, 224)

        # ── Bio Branch: YCrCb clip → rPPG 3D CNN ────────────────────────────
        bio_frames = []
        for f in frames:
            ycrcb = cv2.cvtColor(f, cv2.COLOR_BGR2YCrCb)
            ycrcb = cv2.resize(ycrcb, (self.BIO_SIZE, self.BIO_SIZE),
                               interpolation=cv2.INTER_LINEAR)
            bio_frames.append(ycrcb)

        # (T, H, W, 3) uint8 → (T, 3, H, W) float [0, 1]
        bio_array = np.stack(bio_frames, axis=0).astype(np.float32) / 255.0
        bio_input = torch.from_numpy(bio_array).permute(0, 3, 1, 2)  # (T, 3, H, W)

        return spatial_input, bio_input, torch.tensor(label, dtype=torch.float32)


print('DeepfakeDataset defined.')

In [ ]:
swin_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


def collect_npy(base_dir):
    '''Collect .npy clip paths and labels from real/ and fake/ subfolders.'''
    paths, labels = [], []
    for label_name, label_val in [('real', 0), ('fake', 1)]:
        folder = os.path.join(base_dir, label_name)
        if not os.path.exists(folder):
            continue
        for f in sorted(glob.glob(os.path.join(folder, '*.npy'))):
            paths.append(f)
            labels.append(label_val)
    return paths, labels


def get_video_id(npy_path):
    '''
    Trích video_id từ tên clip.
    Format: "{video_id}_clip{NN}.npy" → "{video_id}"
    Dùng rfind để tránh false-split nếu video_id có "_clip" trong tên.
    '''
    name = os.path.splitext(os.path.basename(npy_path))[0]
    idx  = name.rfind('_clip')
    return name[:idx] if idx != -1 else name


# ── Collect FF++ data ──────────────────────────────────────────────────────────
ff_paths, ff_labels = collect_npy(Config.FF_CLIPS)
print(f'FF++ total: {len(ff_paths)} clips '
      f'(Real: {ff_labels.count(0)}, Fake: {ff_labels.count(1)})')

# ── Split by video ID — tránh data leakage (frames cùng video cùng split) ─────
video_ids = sorted(set(get_video_id(p) for p in ff_paths))
random.shuffle(video_ids)
split = int(len(video_ids) * 0.8)
train_vids = set(video_ids[:split])
val_vids   = set(video_ids[split:])

train_paths, train_labels = [], []
val_paths,   val_labels   = [], []

for p, l in zip(ff_paths, ff_labels):
    if get_video_id(p) in train_vids:
        train_paths.append(p);  train_labels.append(l)
    else:
        val_paths.append(p);    val_labels.append(l)

print(f'FF++ Train: {len(train_paths)} clips '
      f'(Real: {train_labels.count(0)}, Fake: {train_labels.count(1)})')
print(f'FF++ Val:   {len(val_paths)} clips '
      f'(Real: {val_labels.count(0)}, Fake: {val_labels.count(1)})')

# ── Celeb-DF-v2: cross-dataset test (không chạm vào khi train/val) ────────────
celeb_paths, celeb_labels = collect_npy(Config.CELEB_CLIPS)
print(f'\nCeleb-DF-v2 Test: {len(celeb_paths)} clips '
      f'(Real: {celeb_labels.count(0)}, Fake: {celeb_labels.count(1)})')

In [ ]:
train_dataset = DeepfakeDataset(train_paths, train_labels, swin_transform, augment=True)
val_dataset   = DeepfakeDataset(val_paths,   val_labels,   swin_transform, augment=False)
test_dataset  = DeepfakeDataset(celeb_paths, celeb_labels, swin_transform, augment=False)

# WeightedRandomSampler để cân bằng class imbalance (FF++ có nhiều fake hơn real)
counts         = np.bincount(train_labels)
sample_weights = [1.0 / counts[l] for l in train_labels]
sampler        = WeightedRandomSampler(sample_weights, len(sample_weights),
                                       replacement=True)

train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE,
                          sampler=sampler, num_workers=2,
                          pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=Config.BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

print(f'DataLoaders ready:')
print(f'  Train: {len(train_loader)} batches × {Config.BATCH_SIZE}')
print(f'  Val:   {len(val_loader)} batches')
print(f'  Test:  {len(test_loader)} batches (Celeb-DF-v2)')

---
## Section 3: Model Architecture — DeepfakeFusionModel V5

### rPPGNetwork3D — outputs a TOKEN SEQUENCE, not a single vector
3D CNN xử lý chuỗi T frame YCrCb. **Khác V4**: thay vì `AdaptiveAvgPool3d((1,1,1))` pool
hết chiều thời gian, block cuối dùng `AdaptiveAvgPool3d((BIO_TOKENS,1,1))` — chỉ pool
spatial, **giữ lại `BIO_TOKENS=4` token theo thời gian**. Đây là sửa đổi bắt buộc để
Cross-Attention có ý nghĩa toán học thật sự (xem giải thích ở đầu notebook).

### CrossAttentionFusion — true multi-token cross-attention
`nn.MultiheadAttention` với `num_heads = Config.ATTN_HEADS = 4`.
- **Query**: 1 token từ spatial branch (Swin) — *"vùng mặt này có gì bất thường?"*
- **Key/Value**: 4 token từ bio branch (rPPG) — *"tín hiệu sinh học nói gì ở 4 đoạn thời gian khác nhau?"*

Vì `K, V` có **4 token** (không phải 1 như V4), `softmax(QK^T/√d_k)` cho ra một phân phối
thật sự trên 4 vị trí — không suy biến về hằng số 1. Trọng số attention giờ có thể được
trích xuất và visualize (xem Section 6.5) để kiểm chứng thực nghiệm.

In [ ]:
class rPPGNetwork3D(nn.Module):
    '''
    Temporal Physiological Feature Extractor — TOKEN SEQUENCE output.

    Input:  (B, T, 3, 64, 64) — YCrCb face clips (T consecutive frames)
    Output: (B, BIO_TOKENS, BIO_DIM) — sequence of temporal physiological tokens

    FIX vs V4: V4's AdaptiveAvgPool3d((1,1,1)) collapsed the entire clip into ONE
    vector, leaving nothing for cross-attention to attend over (K/V sequence length
    was 1, forcing softmax([s])=[1] regardless of s — a mathematical identity that
    makes the attention mechanism degenerate into a fixed linear map of bio_feat,
    with zero gradient flowing to W^Q / W^K).

    V5 keeps BIO_TOKENS=4 separate temporal positions by pooling ONLY the spatial
    dimensions in the final block, so K and V each have 4 real positions and the
    softmax distribution is non-trivial.

    Block design:
      Block 1: (B, 3, T,    64, 64) → (B, 32, T,    32, 32)  [pool spatial only]
      Block 2: (B, 32, T,   32, 32) → (B, 64, T//2, 16, 16)  [pool spatial + temporal]
      Block 3: (B, 64, T//2, ...)   → (B, BIO_DIM, BIO_TOKENS, 1, 1) [pool spatial only]
    '''
    def __init__(self, bio_dim=Config.BIO_DIM, bio_tokens=Config.BIO_TOKENS):
        super().__init__()
        self.bio_tokens = bio_tokens
        self.features = nn.Sequential(
            # Block 1: chỉ pool spatial — giữ full temporal resolution
            nn.Conv3d(3, 32, kernel_size=(1, 3, 3), padding=(0, 1, 1)),
            nn.BatchNorm3d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2)),

            # Block 2: pool cả temporal và spatial (T: 8 → 4)
            nn.Conv3d(32, 64, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=(2, 2, 2), stride=(2, 2, 2)),

            # Block 3: deep features — pool CHỈ spatial, GIỮ bio_tokens vị trí thời gian
            nn.Conv3d(64, bio_dim, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.BatchNorm3d(bio_dim),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool3d((bio_tokens, 1, 1))   # ← FIX: giữ chiều thời gian
        )

    def forward(self, x):
        # x: (B, T, 3, H, W) — dataset format
        # → (B, 3, T, H, W) — 3D conv expects channel-first
        x = x.permute(0, 2, 1, 3, 4).contiguous()
        feat = self.features(x)                  # (B, bio_dim, bio_tokens, 1, 1)
        feat = feat.squeeze(-1).squeeze(-1)       # (B, bio_dim, bio_tokens)
        return feat.permute(0, 2, 1).contiguous() # (B, bio_tokens, bio_dim) — token sequence

In [ ]:
class CrossAttentionFusion(nn.Module):
    '''
    Multi-Head Cross-Attention Fusion (Vaswani et al., 2017) — V5, mathematically valid.

    Spatial features (Swin) → Query Q   : 1 token, "tôi thấy gì ở vùng mặt này?"
    Bio features (rPPG 3D)  → Key/Value : BIO_TOKENS=4 tokens, "tín hiệu sinh học nói
                                           gì ở từng đoạn thời gian?"

    FIX vs V4: K/V trong V4 chỉ có 1 token → softmax([s]) ≡ [1] với MỌI giá trị s
    (đẳng thức toán học, không phải xấp xỉ) → attention luôn trả về đúng V, hoàn
    toàn không phụ thuộc Q, và W^Q/W^K không nhận gradient từ task loss.

    V5: K/V có 4 token thật sự → softmax(QK^T/√d_k) là một phân phối có ý nghĩa
    trên 4 vị trí thời gian, có thể trích xuất để visualize (xem evaluate_model
    và Section 6.5).
    '''
    def __init__(self, spatial_dim=Config.SPATIAL_DIM, bio_dim=Config.BIO_DIM,
                 num_heads=Config.ATTN_HEADS, dropout=0.1):
        super().__init__()
        assert spatial_dim % num_heads == 0, (
            f'spatial_dim ({spatial_dim}) phải chia hết cho num_heads ({num_heads})')

        # Project mỗi bio token (64-d) lên spatial_dim (768-d) để làm K, V
        # Linear áp dụng độc lập cho từng token trong sequence (B, BIO_TOKENS, bio_dim)
        self.bio_proj = nn.Sequential(
            nn.Linear(bio_dim, spatial_dim),
            nn.LayerNorm(spatial_dim)
        )

        # Standard multi-head cross-attention
        # Q: spatial feature (1 token) | K, V: projected bio tokens (BIO_TOKENS tokens)
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=spatial_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.layer_norm = nn.LayerNorm(spatial_dim)
        self.dropout    = nn.Dropout(dropout)

    def forward(self, spatial_feat, bio_tokens, return_attn=False):
        '''
        spatial_feat : (B, 768)              — từ Swin Transformer
        bio_tokens   : (B, BIO_TOKENS, 64)    — từ rPPG 3D CNN (sequence!)
        Returns      : (B, 768) fused features
                       [+ (B, BIO_TOKENS) attention weights nếu return_attn=True]
        '''
        Q  = spatial_feat.unsqueeze(1)        # (B, 1, 768)            — 1 query token
        KV = self.bio_proj(bio_tokens)        # (B, BIO_TOKENS, 768)   — 4 key/value tokens

        attended, attn_w = self.cross_attn(
            Q, KV, KV, need_weights=True, average_attn_weights=True)
        # attended: (B, 1, 768) | attn_w: (B, 1, BIO_TOKENS) — averaged across 4 heads

        attended = attended.squeeze(1)        # (B, 768)
        fused    = self.layer_norm(spatial_feat + self.dropout(attended))

        if return_attn:
            return fused, attn_w.squeeze(1)   # attn_w: (B, BIO_TOKENS), sums to 1 per sample
        return fused

In [ ]:
class DeepfakeFusionModelV5(nn.Module):
    '''
    Deepfake Detection Model V5 — true multi-token Cross-Attention.

    Pipeline:
        RGB frame (middle of T-frame clip)
          → Swin-Tiny (pretrained, ImageNet)
          → 768-d spatial feature (1 query token)
                    \\
                     CrossAttentionFusion (4 heads, K/V = 4 temporal tokens)
                    /                    → MLP → Sigmoid
        YCrCb clip (T frames)
          → rPPGNetwork3D (3D CNN)
          → (BIO_TOKENS=4, 64) temporal physiological token sequence
    '''
    def __init__(self, pretrained_swin=True):
        super().__init__()
        self.swin = timm.create_model(
            'swin_tiny_patch4_window7_224',
            pretrained=pretrained_swin,
            num_classes=0
        )
        self.rppg_net = rPPGNetwork3D(
            bio_dim=Config.BIO_DIM, bio_tokens=Config.BIO_TOKENS)
        self.cross_attention = CrossAttentionFusion(
            spatial_dim=Config.SPATIAL_DIM,
            bio_dim=Config.BIO_DIM,
            num_heads=Config.ATTN_HEADS,
            dropout=0.1
        )
        self.classifier = nn.Sequential(
            nn.Linear(Config.SPATIAL_DIM, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(Config.DROPOUT),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(Config.DROPOUT),

            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, spatial_input, bio_input, return_attn=False):
        spatial_feat = self.swin(spatial_input)           # (B, 768)
        bio_tokens   = self.rppg_net(bio_input)            # (B, BIO_TOKENS, 64)

        if return_attn:
            fused, attn_w = self.cross_attention(
                spatial_feat, bio_tokens, return_attn=True)
            return self.classifier(fused), attn_w           # attn_w: (B, BIO_TOKENS)

        fused = self.cross_attention(spatial_feat, bio_tokens)
        return self.classifier(fused)                        # (B, 1)


# ── Verify model + verify attention is non-degenerate ──────────────────────────
model = DeepfakeFusionModelV5(pretrained_swin=True).to(device)

dummy_s = torch.randn(4, 3, 224, 224).to(device)
dummy_b = torch.randn(4, Config.T, 3, 64, 64).to(device)
with torch.no_grad():
    out = model(dummy_s, dummy_b)
    out_attn, attn_w = model(dummy_s, dummy_b, return_attn=True)

assert out.shape == (4, 1), f'Unexpected output shape: {out.shape}'
assert attn_w.shape == (4, Config.BIO_TOKENS), f'Unexpected attn shape: {attn_w.shape}'

print(f'Forward pass OK. Output shape: {out.shape}')
print(f'Attention weights shape: {attn_w.shape}  (B={4}, BIO_TOKENS={Config.BIO_TOKENS})')
print(f'Attention weights sum per sample (should ≈ 1.0): '
      f'{attn_w.sum(dim=1).cpu().numpy()}')
print(f'Sample attention distribution (row 0): {attn_w[0].cpu().numpy().round(4)}')

# Sanity check: weights must NOT be degenerate (e.g. all equal to 1/BIO_TOKENS
# AND identical across random different dummy spatial inputs would indicate Q
# has no effect — with random untrained weights some variation is expected,
# but the key fix here is structural: K/V sequence length > 1 makes softmax
# mathematically capable of producing a non-trivial distribution.)
print(f'\nStd across the 4 token weights (row 0): {attn_w[0].std().item():.6f}')
print('(Non-zero std confirms softmax is NOT mathematically forced to [1] anymore.)')

total_p = sum(p.numel() for p in model.parameters())
print(f'\nParameter count:')
print(f'  Swin Transformer : {sum(p.numel() for p in model.swin.parameters()):>12,}')
print(f'  rPPG 3D CNN      : {sum(p.numel() for p in model.rppg_net.parameters()):>12,}')
print(f'  CrossAttention   : {sum(p.numel() for p in model.cross_attention.parameters()):>12,}')
print(f'  Classifier       : {sum(p.numel() for p in model.classifier.parameters()):>12,}')
print(f'  TOTAL            : {total_p:>12,}')

del dummy_s, dummy_b, out, out_attn, attn_w
torch.cuda.empty_cache()

---
## Section 4 & 5: Training Configuration & Loop

In [ ]:
class LabelSmoothingBCELoss(nn.Module):
    '''BCELoss + label smoothing để giảm overconfidence.'''
    def __init__(self, smoothing=Config.LABEL_SMOOTHING):
        super().__init__()
        self.s   = smoothing
        self.bce = nn.BCELoss()

    def forward(self, pred, target):
        target_smooth = target * (1 - self.s) + 0.5 * self.s
        return self.bce(pred, target_smooth)


class EarlyStopping:
    '''Dừng training khi val AUC không cải thiện sau `patience` epochs.'''
    def __init__(self, patience=Config.EARLY_STOP_PATIENCE, min_delta=1e-4):
        self.patience  = patience
        self.min_delta = min_delta
        self.counter   = 0
        self.best      = None
        self.stop      = False

    def __call__(self, score):
        if self.best is None:
            self.best = score
        elif score < self.best + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        else:
            self.best   = score
            self.counter = 0


def compute_eer(y_true, y_scores):
    '''Equal Error Rate (%) via brentq — chính xác hơn nanargmin.'''
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    eer = brentq(lambda x: 1.0 - x - interp1d(fpr, tpr)(x), 0.0, 1.0)
    return eer * 100


print('Training utilities ready.')

In [ ]:
os.makedirs(Config.SAVE_DIR, exist_ok=True)

criterion = LabelSmoothingBCELoss()

# Differential LR: Swin pretrained → nhỏ; các layer mới → lớn hơn
swin_params  = list(model.swin.parameters())
other_params = [p for n, p in model.named_parameters()
                if not n.startswith('swin')]

optimizer = optim.AdamW([
    {'params': swin_params,  'lr': Config.LR_SWIN},
    {'params': other_params, 'lr': Config.LR_OTHER}
], weight_decay=Config.WEIGHT_DECAY)

scheduler       = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=Config.EPOCHS, eta_min=1e-7)
early_stopping  = EarlyStopping()

history = defaultdict(list)
best_val_auc = 0.0
best_epoch   = 0
best_path    = ''

print(f'TRAINING START  |  {Config.EPOCHS} epochs  |  batch_size={Config.BATCH_SIZE}')
print(f'Swin LR: {Config.LR_SWIN}  |  Other LR: {Config.LR_OTHER}')
print('=' * 90)

for epoch in range(1, Config.EPOCHS + 1):
    t0 = time.time()

    # ── TRAIN ─────────────────────────────────────────────────────────────────
    model.train()
    tr_loss, tr_corr, tr_total = 0.0, 0, 0

    pbar = tqdm(train_loader, desc=f'Ep {epoch:02d} [Train]', leave=False)
    for imgs, bios, labels in pbar:
        imgs, bios, labels = imgs.to(device), bios.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs, bios).squeeze(1)   # FIX: squeeze(1) not squeeze()
        loss    = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), Config.GRAD_CLIP)
        optimizer.step()

        tr_loss  += loss.item() * imgs.size(0)
        preds     = (outputs > 0.5).float()
        tr_corr  += (preds == labels).sum().item()
        tr_total += labels.size(0)
        pbar.set_postfix(loss=f'{loss.item():.4f}',
                         acc=f'{tr_corr/tr_total:.4f}')

    tr_loss /= tr_total
    tr_acc   = tr_corr / tr_total

    # ── VALIDATE ──────────────────────────────────────────────────────────────
    model.eval()
    va_loss, va_corr, va_total = 0.0, 0, 0
    all_labels, all_probs = [], []

    with torch.no_grad():
        for imgs, bios, labels in tqdm(val_loader, desc=f'Ep {epoch:02d} [Val]',
                                       leave=False):
            imgs, bios, labels = imgs.to(device), bios.to(device), labels.to(device)
            outputs = model(imgs, bios).squeeze(1)   # FIX: squeeze(1)
            loss    = criterion(outputs, labels)

            va_loss  += loss.item() * imgs.size(0)
            preds     = (outputs > 0.5).float()
            va_corr  += (preds == labels).sum().item()
            va_total += labels.size(0)
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(outputs.cpu().numpy())

    va_loss /= va_total
    va_acc   = va_corr / va_total
    va_auc   = roc_auc_score(all_labels, all_probs)
    va_eer   = compute_eer(all_labels, all_probs)

    scheduler.step()
    lr_now = scheduler.get_last_lr()[0]

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(va_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(va_acc)
    history['val_auc'].append(va_auc)
    history['val_eer'].append(va_eer)
    history['lr'].append(lr_now)

    marker = ''
    if va_auc > best_val_auc:
        best_val_auc = va_auc
        best_epoch   = epoch
        best_path    = os.path.join(Config.SAVE_DIR, 'best_model_v5.pth')
        torch.save(model.state_dict(), best_path)
        marker = '  << BEST'

    if epoch % 5 == 0:
        ckpt = os.path.join(Config.SAVE_DIR, f'v5_ep{epoch:02d}.pth')
        torch.save(model.state_dict(), ckpt)

    elapsed = time.time() - t0
    print(f'Ep {epoch:02d}/{Config.EPOCHS} | '
          f'TrL {tr_loss:.4f} TrA {tr_acc:.4f} | '
          f'VaL {va_loss:.4f} VaA {va_acc:.4f} AUC {va_auc:.4f} EER {va_eer:.2f}% | '
          f'LR {lr_now:.2e} | {elapsed:.0f}s{marker}')

    early_stopping(va_auc)
    if early_stopping.stop:
        print(f'\nEarly stopping triggered at epoch {epoch}.')
        break

    torch.cuda.empty_cache()

print(f'\nBest: Epoch {best_epoch}  |  Val AUC = {best_val_auc:.4f}')
print(f'Saved to: {best_path}')

In [ ]:
# Training curves
epochs_r = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Training History — DeepfakeFusionModel V5', fontsize=14, fontweight='bold')

axes[0,0].plot(epochs_r, history['train_loss'], 'o-', color='#FF6B6B',
               label='Train', markersize=4)
axes[0,0].plot(epochs_r, history['val_loss'],   's-', color='#4ECDC4',
               label='Val',   markersize=4)
axes[0,0].set(title='Loss', xlabel='Epoch', ylabel='Loss'); axes[0,0].legend()

axes[0,1].plot(epochs_r, history['train_acc'], 'o-', color='#FF6B6B',
               label='Train', markersize=4)
axes[0,1].plot(epochs_r, history['val_acc'],   's-', color='#4ECDC4',
               label='Val',   markersize=4)
axes[0,1].set(title='Accuracy', xlabel='Epoch', ylabel='Accuracy'); axes[0,1].legend()

axes[1,0].plot(epochs_r, history['val_auc'], 's-', color='#45B7D1',
               linewidth=2, markersize=4, label='Val AUC')
axes[1,0].set(title='Validation AUC-ROC', xlabel='Epoch', ylabel='AUC')
axes[1,0].legend()

axes[1,1].plot(epochs_r, history['val_eer'], 's-', color='#96CEB4',
               linewidth=2, markersize=4, label='Val EER')
axes[1,1].set(title='Validation EER (lower=better)', xlabel='Epoch', ylabel='EER (%)')
axes[1,1].legend()

plt.tight_layout()
plt.savefig(os.path.join(Config.SAVE_DIR, 'training_curves_v5.png'),
            dpi=150, bbox_inches='tight')
plt.show()

---
## Section 6: Evaluation — FF++ Validation

In [ ]:
def evaluate_model(model, dataloader, dataset_name='Dataset'):
    '''Full evaluation: metrics + Confusion Matrix + ROC + Score Distribution.'''
    model.eval()
    all_labels, all_probs = [], []

    with torch.no_grad():
        for imgs, bios, labels in tqdm(dataloader, desc=f'Eval: {dataset_name}'):
            imgs, bios = imgs.to(device), bios.to(device)
            outputs    = model(imgs, bios).squeeze(1)  # FIX: squeeze(1)
            all_labels.extend(labels.numpy())
            all_probs.extend(outputs.cpu().numpy())

    y_true = np.array(all_labels)
    y_prob = np.array(all_probs)
    y_pred = (y_prob > 0.5).astype(int)

    acc  = accuracy_score(y_true, y_pred)
    auc  = roc_auc_score(y_true, y_prob)
    eer  = compute_eer(y_true, y_prob)
    f1   = f1_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred)

    print(f'\n{"=" * 52}')
    print(f'RESULTS — {dataset_name}')
    print(f'{"=" * 52}')
    print(f'  Accuracy  : {acc * 100:.2f}%')
    print(f'  AUC-ROC   : {auc:.4f}')
    print(f'  EER       : {eer:.2f}%')
    print(f'  F1-Score  : {f1:.4f}')
    print(f'  Precision : {prec:.4f}')
    print(f'  Recall    : {rec:.4f}')
    print()
    print(classification_report(y_true, y_pred, target_names=['Real', 'Fake']))

    # Plots
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'Evaluation — {dataset_name}', fontsize=13, fontweight='bold')

    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Real','Fake'], yticklabels=['Real','Fake'], ax=axes[0])
    axes[0].set(xlabel='Predicted', ylabel='Actual', title='Confusion Matrix')

    fpr, tpr, _ = roc_curve(y_true, y_prob)
    axes[1].plot(fpr, tpr, color='#FF6B6B', linewidth=2, label=f'AUC={auc:.4f}')
    axes[1].plot([0,1],[0,1], 'k--', alpha=0.5)
    axes[1].fill_between(fpr, tpr, alpha=0.1, color='#FF6B6B')
    axes[1].set(xlabel='FPR', ylabel='TPR', title='ROC Curve')
    axes[1].legend(loc='lower right')

    axes[2].hist(y_prob[y_true==0], bins=50, alpha=0.7, color='#4ECDC4',
                 label='Real', density=True)
    axes[2].hist(y_prob[y_true==1], bins=50, alpha=0.7, color='#FF6B6B',
                 label='Fake', density=True)
    axes[2].axvline(0.5, color='black', ls='--', alpha=0.7, label='Threshold 0.5')
    axes[2].set(xlabel='P(Fake)', ylabel='Density', title='Score Distribution')
    axes[2].legend()

    plt.tight_layout()
    safe_name = dataset_name.lower().replace(' ','_').replace('(','').replace(')','')
    plt.savefig(os.path.join(Config.SAVE_DIR, f'eval_{safe_name}.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

    return {'accuracy': acc, 'auc': auc, 'eer': eer,
            'f1': f1, 'precision': prec, 'recall': rec}


# Load best checkpoint
model.load_state_dict(torch.load(best_path, map_location=device))
print(f'Loaded best model from epoch {best_epoch}')

ff_results = evaluate_model(model, val_loader, 'FaceForensics++ Val (C23)')

---
## Section 6.5: Cross-Attention Weight Analysis (empirical proof of fix)

Mục này tồn tại vì lý do cụ thể: chứng minh bằng thực nghiệm rằng `softmax` trong
`CrossAttentionFusion` **không còn suy biến** về hằng số `[1]` như ở V4.

Với V4 (`K,V` chỉ có 1 token), trọng số attention luôn **chính xác bằng 1.0** —
không có gì để visualize, vì nó là một hằng số toán học. Với V5 (`K,V` có
`BIO_TOKENS=4` token), trọng số attention là một phân phối có thể khác nhau giữa
các sample, giữa các epoch huấn luyện, và (lý tưởng) giữa class Real vs Fake.

Nếu biểu đồ dưới đây cho thấy **phương sai khác 0** và **phân phối không đồng nhất
1/4 = 0.25 cho mọi token**, đó là bằng chứng cho thấy attention thực sự "hoạt động"
chứ không phải một phép chiếu tuyến tính cố định bị ngụy trang.

In [ ]:
@torch.no_grad()
def extract_attention_weights(model, dataloader, max_samples=500):
    '''
    Chạy forward với return_attn=True trên val set, thu thập:
      - attn_weights: (N, BIO_TOKENS) — trọng số attention mỗi sample
      - labels:       (N,)            — 0=real, 1=fake
    '''
    model.eval()
    all_attn, all_labels = [], []
    n_collected = 0

    for imgs, bios, labels in tqdm(dataloader, desc='Extracting attention'):
        imgs, bios = imgs.to(device), bios.to(device)
        _, attn_w = model(imgs, bios, return_attn=True)   # (B, BIO_TOKENS)
        all_attn.append(attn_w.cpu().numpy())
        all_labels.append(labels.numpy())
        n_collected += imgs.size(0)
        if n_collected >= max_samples:
            break

    return np.concatenate(all_attn, axis=0), np.concatenate(all_labels, axis=0)


attn_weights, attn_labels = extract_attention_weights(model, val_loader)

print(f'Collected attention weights: {attn_weights.shape}')
print(f'Mean weight per token (all samples): {attn_weights.mean(axis=0).round(4)}')
print(f'Std  weight per token (all samples): {attn_weights.std(axis=0).round(4)}')
print(f'Per-sample std (mean over samples): {attn_weights.std(axis=1).mean():.4f}')
print(f'  → Nếu std ≈ 0 across all samples: attention gần như đồng nhất (uniform),')
print(f'    vẫn hợp lệ về toán học nhưng yếu về mặt "dynamic selectivity".')
print(f'  → Nếu std > 0 rõ rệt: model đang phân biệt các token thời gian khác nhau.')

# ── So sánh phân phối attention: Real vs Fake ───────────────────────────────────
real_attn = attn_weights[attn_labels == 0]
fake_attn = attn_weights[attn_labels == 1]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Cross-Attention Weight Distribution per Temporal Token',
             fontsize=13, fontweight='bold')

x = np.arange(Config.BIO_TOKENS)
width = 0.35
axes[0].bar(x - width/2, real_attn.mean(axis=0), width,
            yerr=real_attn.std(axis=0), label='Real', color='#4ECDC4', capsize=4)
axes[0].bar(x + width/2, fake_attn.mean(axis=0), width,
            yerr=fake_attn.std(axis=0), label='Fake', color='#FF6B6B', capsize=4)
axes[0].axhline(1/Config.BIO_TOKENS, color='gray', ls='--', alpha=0.6,
                label=f'Uniform (1/{Config.BIO_TOKENS})')
axes[0].set(xlabel='Temporal Token Index', ylabel='Mean Attention Weight',
           title='Average attention per class', xticks=x)
axes[0].legend()

axes[1].hist(attn_weights.std(axis=1), bins=30, color='#45B7D1', alpha=0.8)
axes[1].axvline(0, color='red', ls='--', label='Degenerate (V4 bug)')
axes[1].set(xlabel='Per-sample Std across 4 tokens', ylabel='Count',
           title='Distribution of attention "sharpness"')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(Config.SAVE_DIR, 'attention_analysis_v5.png'),
            dpi=150, bbox_inches='tight')
plt.show()

print('\nNote khi viết paper: nếu Real vs Fake cho thấy pattern attention khác')
print('biệt rõ rệt (vd: Fake tập trung mạnh vào 1-2 token cụ thể, Real phân bố đều')
print('hơn), đây là bằng chứng thực nghiệm trực tiếp hỗ trợ claim "the network')
print('focuses on regions where biological liveness is disrupted" trong Conclusion.')

---
## Section 7: Cross-Dataset Evaluation — Celeb-DF-v2

Model **chưa từng thấy** Celeb-DF-v2 — đây là test generalization thực sự.

In [ ]:
celeb_results = evaluate_model(model, test_loader, 'Celeb-DF-v2 (Cross-Dataset)')

print('\n' + '=' * 72)
print('COMPARISON TABLE')
print('=' * 72)
print(f'{"Metric":<14} {"FF++ Val":>12} {"Celeb-DF-v2":>14}')
print('-' * 44)
for k in ['accuracy', 'auc', 'eer', 'f1', 'precision', 'recall']:
    ff_v    = ff_results[k]
    celeb_v = celeb_results[k]
    if k in ('accuracy', 'eer'):
        print(f'{k:<14} {ff_v*100:>11.2f}%  {celeb_v*100:>13.2f}%')
    else:
        print(f'{k:<14} {ff_v:>12.4f}  {celeb_v:>14.4f}')

---
## Section 8: Ablation Study (Fair Comparison)

**Thiết kế fair hơn V3:**
- Tất cả variants: `ABLATION_EPOCHS` epochs, cùng scheduler, cùng gradient clip
- Swin-based variants: differential LR (Swin=2e-5, other=1e-4) nhất quán
- rPPG-only variant: uniform LR=1e-4 (không có Swin)
- Đây là quick comparison — explicitly noted, không dùng để claim absolute numbers

| Variant | Spatial | Temporal Bio | Fusion |
|---|---|---|---|
| **A** Swin Only | Swin-Tiny | ✗ | — |
| **B** rPPG Only | ✗ | rPPGNetwork3D | — |
| **C** Concat | Swin-Tiny | rPPGNetwork3D | Cat(768+64) → MLP |
| **D** Full (V5) | Swin-Tiny | rPPGNetwork3D | Multi-Head CrossAttn |

In [ ]:
class SwinOnlyModel(nn.Module):
    '''Ablation A: Spatial features only.'''
    def __init__(self):
        super().__init__()
        self.swin = timm.create_model(
            'swin_tiny_patch4_window7_224', pretrained=True, num_classes=0)
        self.classifier = nn.Sequential(
            nn.Linear(768, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 1), nn.Sigmoid())

    def forward(self, spatial_input, bio_input):
        return self.classifier(self.swin(spatial_input))


class rPPGOnlyModel(nn.Module):
    '''
    Ablation B: Temporal physiological features only.
    rppg_net now returns a (B, BIO_TOKENS, BIO_DIM) token sequence (V5 change) —
    mean-pool across tokens to get a single (B, BIO_DIM) vector for this baseline,
    since there is no cross-attention here to consume the full sequence.
    '''
    def __init__(self):
        super().__init__()
        self.rppg_net = rPPGNetwork3D(
            bio_dim=Config.BIO_DIM, bio_tokens=Config.BIO_TOKENS)
        self.classifier = nn.Sequential(
            nn.Linear(Config.BIO_DIM, 128), nn.BatchNorm1d(128),
            nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 1), nn.Sigmoid())

    def forward(self, spatial_input, bio_input):
        tokens = self.rppg_net(bio_input)      # (B, BIO_TOKENS, BIO_DIM)
        pooled = tokens.mean(dim=1)             # (B, BIO_DIM) — simple temporal average
        return self.classifier(pooled)


class ConcatFusionModel(nn.Module):
    '''
    Ablation C: Concat fusion baseline (no attention at all).
    Bio tokens mean-pooled to a single vector, then concatenated with spatial —
    this is the "naive concatenation" baseline the paper explicitly argues against.
    '''
    def __init__(self):
        super().__init__()
        self.swin     = timm.create_model(
            'swin_tiny_patch4_window7_224', pretrained=True, num_classes=0)
        self.rppg_net = rPPGNetwork3D(
            bio_dim=Config.BIO_DIM, bio_tokens=Config.BIO_TOKENS)
        in_dim = Config.SPATIAL_DIM + Config.BIO_DIM
        self.classifier = nn.Sequential(
            nn.Linear(in_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 1), nn.Sigmoid())

    def forward(self, spatial_input, bio_input):
        s = self.swin(spatial_input)
        b_tokens = self.rppg_net(bio_input)     # (B, BIO_TOKENS, BIO_DIM)
        b = b_tokens.mean(dim=1)                 # (B, BIO_DIM)
        return self.classifier(torch.cat((s, b), dim=1))


print('Ablation model variants defined (V5: rPPG branch outputs token sequences,'
      ' mean-pooled for non-attention baselines).')

In [ ]:
def ablation_train_eval(model_cls, variant_name):
    '''
    Train ablation variant với điều kiện nhất quán.
    Fair so với V5 full model:
      - Cùng differential LR (Swin vs others) nếu có Swin
      - Cùng CosineAnnealingLR schedule
      - Cùng gradient clipping
    Note: ít epochs hơn full training — kết quả là indicative, không absolute.
    '''
    print(f'\n{"─"*50}\n{variant_name}\n{"─"*50}')
    m = model_cls().to(device)
    crit = LabelSmoothingBCELoss()

    # Differential LR nhất quán — nếu có Swin thì Swin LR thấp hơn
    try:
        swin_p  = list(m.swin.parameters())
        other_p = [p for n, p in m.named_parameters() if not n.startswith('swin')]
        opt = optim.AdamW([
            {'params': swin_p,  'lr': Config.LR_SWIN},
            {'params': other_p, 'lr': Config.LR_OTHER}
        ], weight_decay=Config.WEIGHT_DECAY)
    except AttributeError:
        # rPPG-only: không có Swin, dùng uniform LR
        opt = optim.AdamW(m.parameters(), lr=Config.LR_OTHER,
                          weight_decay=Config.WEIGHT_DECAY)

    sched = optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=Config.ABLATION_EPOCHS, eta_min=1e-7)

    for ep in range(1, Config.ABLATION_EPOCHS + 1):
        m.train()
        for imgs, bios, labels in tqdm(
                train_loader, desc=f'  Ep{ep:02d}', leave=False):
            imgs, bios, labels = imgs.to(device), bios.to(device), labels.to(device)
            opt.zero_grad()
            out  = m(imgs, bios).squeeze(1)
            loss = crit(out, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), Config.GRAD_CLIP)
            opt.step()
        sched.step()

    # Evaluate on val set
    m.eval()
    v_labels, v_probs = [], []
    with torch.no_grad():
        for imgs, bios, labels in val_loader:
            imgs, bios = imgs.to(device), bios.to(device)
            out = m(imgs, bios).squeeze(1)
            v_labels.extend(labels.numpy())
            v_probs.extend(out.cpu().numpy())

    y_true = np.array(v_labels)
    y_prob = np.array(v_probs)
    acc = accuracy_score(y_true, (y_prob > 0.5).astype(int))
    auc = roc_auc_score(y_true, y_prob)
    eer = compute_eer(y_true, y_prob)
    print(f'  Acc={acc*100:.2f}%  AUC={auc:.4f}  EER={eer:.2f}%')

    del m, opt, sched, crit
    torch.cuda.empty_cache(); gc.collect()
    return {'accuracy': acc, 'auc': auc, 'eer': eer}


print(f'ABLATION STUDY  |  {Config.ABLATION_EPOCHS} epochs per variant')
print(f'(Indicative comparison — full model trained for {Config.EPOCHS} epochs)')
print('=' * 60)

ablation_results = {}
for name, cls in [
    ('A: Swin Only',           SwinOnlyModel),
    ('B: rPPG 3D Only',        rPPGOnlyModel),
    ('C: Concat Fusion',       ConcatFusionModel),
    ('D: Cross-Attn V5 (Full)', DeepfakeFusionModelV5),
]:
    ablation_results[name] = ablation_train_eval(cls, name)

In [ ]:
print('\n' + '=' * 58)
print('ABLATION RESULTS')
print('=' * 58)
print(f'{"Variant":<28} {"Accuracy":>10} {"AUC":>8} {"EER":>8}')
print('-' * 58)
for name, r in ablation_results.items():
    marker = ' ←' if 'Full' in name else ''
    print(f'{name:<28} {r["accuracy"]*100:>9.2f}%  {r["auc"]:>7.4f}  '
          f'{r["eer"]:>6.2f}%{marker}')

# Bar chart
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Ablation Study (Indicative)', fontsize=13, fontweight='bold')
names  = list(ablation_results.keys())
colors = ['#95a5a6', '#e74c3c', '#3498db', '#2ecc71']

for ax, metric, label, scale in [
    (axes[0], 'accuracy', 'Accuracy (%)', 100),
    (axes[1], 'auc',      'AUC-ROC',       1),
    (axes[2], 'eer',      'EER % (↓)',     1),
]:
    vals = [ablation_results[n][metric] * scale for n in names]
    bars = ax.bar(names, vals, color=colors)
    ax.set_title(label)
    ax.tick_params(axis='x', rotation=25)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + (0.5 if scale==100 else 0.002),
                f'{v:.2f}{"%" if scale==100 else ""}',
                ha='center', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(Config.SAVE_DIR, 'ablation_v5.png'),
            dpi=150, bbox_inches='tight')
plt.show()

---
## Section 9: Export & Final Summary

In [ ]:
model.load_state_dict(torch.load(best_path, map_location=device))

final_name    = f'fusion_model_v5_auc{best_val_auc:.3f}_ep{best_epoch:02d}.pth'
final_path    = os.path.join(Config.SAVE_DIR, final_name)
backend_path  = os.path.join(Config.SAVE_DIR, 'fusion_model_v5_best.pth')
torch.save(model.state_dict(), final_path)
torch.save(model.state_dict(), backend_path)

print('=' * 60)
print('TRAINING COMPLETE — FINAL SUMMARY')
print('=' * 60)
total_p = sum(p.numel() for p in model.parameters())
print(f'  Model:            DeepfakeFusionModel V5')
print(f'  Total parameters: {total_p:,}')
print(f'  Best epoch:       {best_epoch}')
print(f'  Best val AUC:     {best_val_auc:.4f}')
print(f'\n  FF++ Val results:')
for k, v in ff_results.items():
    s = f'{v*100:.2f}%' if k in ('accuracy','eer') else f'{v:.4f}'
    print(f'    {k:<12}: {s}')
print(f'\n  Celeb-DF-v2 (Cross-Dataset):')
for k, v in celeb_results.items():
    s = f'{v*100:.2f}%' if k in ('accuracy','eer') else f'{v:.4f}'
    print(f'    {k:<12}: {s}')
print(f'\n  Saved to: {Config.SAVE_DIR}/')
!ls -lh "{Config.SAVE_DIR}/"
print('\nDone!')